# 🎯 PathForge — Learning Path Recommendation System
### Collaborative Filtering with SVD Matrix Factorization
---
**Objective:** Build a personalized learning path recommender for interns using Matrix Factorization (SVD), trained on intern learning patterns and evaluated with NDCG@3.

## Phase 1 — Setup & Data Loading

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

print(" All libraries loaded")

In [ ]:
# ── Loading Dataset ─────────────────────────────────────────────
excel_file = "intern_learning_path_dataset_v2.xlsx"
df = pd.read_excel(excel_file, sheet_name="Intern Dataset")
print(f" Loaded: {df.shape[0]} intern profiles | {df.shape[1]} features")
print(f"\nColumns: {df.columns.tolist()}")
df.head(3)

In [ ]:
ALL_COURSES = [
    'Python Fundamentals', 'SQL & Databases', 'Machine Learning Basics', 'Deep Learning',
    'Data Visualization', 'Cloud Computing (AWS)', 'Docker & Kubernetes', 'Cybersecurity Essentials',
    'System Design', 'Frontend Basics (React)', 'Statistics & Probability', 'NLP & Text Analytics',
    'MLOps Pipelines', 'A/B Testing & Experimentation', 'Data Wrangling & Pandas',
    'Git & Version Control', 'APIs & Microservices', 'Time Series Analysis',
    'Reinforcement Learning', 'Business Intelligence & BI Tools'
]
print(f" Course catalog: {len(ALL_COURSES)} courses")

## Phase 2 — Building User-Course Interaction Matrix
We construct an implicit feedback matrix where:

In [ ]:
rows = []
for _, r in df.iterrows():
    iid = r['intern_id']
    completed = (str(r['completed_courses']).split(', ')
                 if pd.notna(r['completed_courses']) and r['completed_courses'] != 'nan' else [])
    recommended = [r[c] for c in ['recommended_course_1','recommended_course_2','recommended_course_3']
                   if pd.notna(r[c])]
    seen = set()
    for c in ALL_COURSES:
        if c in completed:
            rows.append({'intern_id': iid, 'course': c, 'rating': float(r['engagement_score'])})
            seen.add(c)
        elif c in recommended and c not in seen:
            rows.append({'intern_id': iid, 'course': c, 'rating': 8.5})

interactions_df = pd.DataFrame(rows)
print(f" Interaction matrix built")
print(f"   Interactions : {len(interactions_df):,}")
print(f"   Interns      : {interactions_df['intern_id'].nunique()}")
print(f"   Courses      : {interactions_df['course'].nunique()}")
print(f"   Rating range : {interactions_df['rating'].min():.1f} – {interactions_df['rating'].max():.1f}")
print(f"   Sparsity     : {100*(1 - len(interactions_df)/(1000*20)):.1f}%")
interactions_df.head(8)

## Phase 3 — Train SVD Matrix Factorization Model
SVD decomposes the user-course interaction matrix into latent factor matrices:
```
R ≈ U × Σ × Vᵀ
```
Where **U** = intern latent factors, **V** = course latent factors, **Σ** = singular values.
This captures hidden patterns — e.g. interns with similar learning styles cluster together.

In [ ]:
reader = Reader(rating_scale=(1, 10))
data   = Dataset.load_from_df(interactions_df[['intern_id','course','rating']], reader)

# 80/20 Train/Test split
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

# SVD model
svd = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

print(" SVD Model Trained")
print(f"   Latent factors : 50")
print(f"   Epochs         : 30")
print(f"   Learning rate  : 0.005")
print(f"   Regularization : 0.02")
print(f"   Train samples  : {trainset.n_ratings:,}")

## Phase 4 — Evaluate on Test Set

In [ ]:
predictions = svd.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)
print(f" Test Set Evaluation")
print(f"   RMSE : {rmse:.4f}")
print(f"   MAE  : {mae:.4f}")

In [ ]:
def ndcg_at_3(predicted, actual):
    actual_set = [str(x).strip().lower() for x in actual if pd.notna(x)]
    dcg  = sum(1.0 / np.log2(i+2) for i, p in enumerate(predicted[:3])
               if str(p).strip().lower() in actual_set)
    idcg = sum(1.0 / np.log2(i+2) for i in range(min(3, len(actual_set))))
    return dcg / idcg if idcg > 0 else 0.0

# Training on full data for deployment
full_trainset = data.build_full_trainset()
svd_full = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
svd_full.fit(full_trainset)

all_recs      = pd.concat([df['recommended_course_1'], df['recommended_course_2'], df['recommended_course_3']])
top3_popular  = all_recs.dropna().value_counts().index[:3].tolist()

svd_ndcg_list, baseline_ndcg_list = [], []
for _, r in df.iterrows():
    actual    = [r['recommended_course_1'], r['recommended_course_2'], r['recommended_course_3']]
    completed = (str(r['completed_courses']).split(', ')
                 if pd.notna(r['completed_courses']) and r['completed_courses'] != 'nan' else [])
    scores = [(c, svd_full.predict(r['intern_id'], c).est)
              for c in ALL_COURSES if c not in completed]
    scores.sort(key=lambda x: x[1], reverse=True)
    svd_top3 = [s[0] for s in scores[:3]]
    svd_ndcg_list.append(ndcg_at_3(svd_top3, actual))
    baseline_ndcg_list.append(ndcg_at_3(top3_popular, actual))

mean_svd      = np.mean(svd_ndcg_list) * 100
mean_baseline = np.mean(baseline_ndcg_list) * 100
lift          = mean_svd - mean_baseline

print("=" * 55)
print("  PERFORMANCE EVALUATION — NDCG@3")
print("=" * 55)
print(f"  Baseline (Popularity)   NDCG@3 : {mean_baseline:.2f}%")
print(f"  SVD Matrix Factorization NDCG@3 : {mean_svd:.2f}%")
print(f"  Personalization Lift            : +{lift:.2f}%")
print("=" * 55)

## Phase 5 — Visualizations

In [ ]:
# Chart 1: Model Comparison Bar
fig = go.Figure()
fig.add_trace(go.Bar(x=['Baseline (Popularity)', 'SVD Matrix Factorization'],
                     y=[mean_baseline, mean_svd],
                     marker_color=['#ff2d78','#c8ff00'],
                     text=[f'{mean_baseline:.2f}%', f'{mean_svd:.2f}%'],
                     textposition='outside'))
fig.update_layout(title='NDCG@3 — Baseline vs SVD Model',
                  yaxis=dict(range=[0,100], ticksuffix='%'),
                  template='plotly_dark', height=400)
fig.show()

In [ ]:
# Chart 2: Course Popularity Heatmap by Department
dept_course = pd.DataFrame(0, index=df['department'].unique(), columns=ALL_COURSES[:10])
for _, r in df.iterrows():
    for col in ['recommended_course_1','recommended_course_2','recommended_course_3']:
        if pd.notna(r[col]) and r[col] in dept_course.columns:
            dept_course.loc[r['department'], r[col]] += 1

fig = px.imshow(dept_course, title='Course Recommendations by Department (Heatmap)',
                color_continuous_scale='YlGn', template='plotly_dark', height=500)
fig.show()

In [ ]:
# Chart 3: Skill score distributions
skill_cols = ['python_skill_score','math_stat_score','sql_score','ml_knowledge_score','cloud_infra_score']
labels     = ['Python','Math/Stats','SQL','ML Knowledge','Cloud']
fig = go.Figure()
for col, lbl in zip(skill_cols, labels):
    fig.add_trace(go.Box(y=df[col], name=lbl, boxmean=True))
fig.update_layout(title='Skill Score Distributions Across All Interns',
                  yaxis_title='Score (1–10)', template='plotly_dark', height=450)
fig.show()

In [ ]:
# Chart 4: Engagement score distribution
fig = px.histogram(df, x='engagement_score', nbins=20,
                   title='Engagement Score Distribution',
                   template='plotly_dark', color_discrete_sequence=['#c8ff00'])
fig.show()

## Phase 6 — Sample Recommendation for One Intern

In [ ]:
def get_svd_recommendation(intern_id, model, df, all_courses, top_n=3):
    row = df[df['intern_id'] == intern_id]
    if row.empty:
        print(f"Intern {intern_id} not found.")
        return

    row = row.iloc[0]
    completed = (str(row['completed_courses']).split(', ')
                 if pd.notna(row['completed_courses']) and row['completed_courses'] != 'nan' else [])

    scores = []
    for c in all_courses:
        if c not in completed:
            pred = model.predict(intern_id, c)
            scores.append({'Course': c, 'Predicted Rating': round(pred.est, 4)})

    result = pd.DataFrame(scores).sort_values('Predicted Rating', ascending=False).head(top_n)
    result.insert(0, 'Rank', [f'#{i+1}' for i in range(len(result))])
    result = result.reset_index(drop=True)

    print(f"\n{'='*50}")
    print(f"  SVD Recommendations for {intern_id}")
    print(f"  Department : {row['department']}")
    print(f"  Completed  : {len(completed)} courses")
    print(f"{'='*50}")
    display(result)

    # Radar chart
    skill_vals = [row['python_skill_score'], row['math_stat_score'], row['sql_score'],
                  row['ml_knowledge_score'], row['cloud_infra_score']]
    skill_lbls = ['Python', 'Math/Stats', 'SQL', 'ML Knowledge', 'Cloud']
    r_vals = skill_vals + [skill_vals[0]]
    r_lbls = skill_lbls + [skill_lbls[0]]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=r_vals, theta=r_lbls, fill='toself',
                                   fillcolor='rgba(200,255,0,0.2)',
                                   line=dict(color='#c8ff00', width=3),
                                   name=intern_id))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0,10])),
                      title=f'Skill Profile — {intern_id}', template='plotly_dark')
    fig.show()

    return result

get_svd_recommendation("INT_1000", svd_full, df, ALL_COURSES)

## Phase 7 — Save Model for Deployment

In [ ]:
# Save the trained SVD model
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(svd_full, f)

# Save metadata
metadata = {
    'all_courses'   : ALL_COURSES,
    'rmse'          : round(rmse, 4),
    'mae'           : round(mae, 4),
    'svd_ndcg'      : round(mean_svd, 2),
    'baseline_ndcg' : round(mean_baseline, 2),
    'lift'          : round(lift, 2),
    'n_interactions': len(interactions_df),
    'n_factors'     : 50,
    'n_epochs'      : 30,
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(" svd_model.pkl saved")
print(" model_metadata.json saved")
print("\nThe Flask backend (app.py) loads svd_model.pkl to serve predictions via API.")
print("The frontend (index.html) calls the API and visualizes recommendations.")